# Indian ANPR — Dataset Exploration
Quick EDA before training: dataset stats, sample images, OCR sanity check.

In [ ]:
import sys
sys.path.insert(0, '..')

import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

RAW_DIR = Path('../data/raw')
images = sorted(RAW_DIR.glob('*.jpg')) + sorted(RAW_DIR.glob('*.png'))
print(f'Total raw images: {len(images)}')

In [ ]:
# Show a grid of sample images
import random
sample = random.sample(images, min(16, len(images)))

fig, axes = plt.subplots(4, 4, figsize=(16, 8))
for ax, p in zip(axes.flat, sample):
    img = cv2.imread(str(p))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    ax.imshow(img)
    ax.set_title(p.name[:20], fontsize=8)
    ax.axis('off')
plt.suptitle('Sample raw images', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Image size distribution
sizes = []
for p in images:
    img = cv2.imread(str(p))
    if img is not None:
        sizes.append(img.shape[:2])

heights = [s[0] for s in sizes]
widths  = [s[1] for s in sizes]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(widths, bins=30, color='steelblue', edgecolor='white')
axes[0].set_title('Image widths')
axes[0].set_xlabel('px')
axes[1].hist(heights, bins=30, color='coral', edgecolor='white')
axes[1].set_title('Image heights')
axes[1].set_xlabel('px')
plt.suptitle(f'Size distribution — {len(sizes)} images')
plt.tight_layout()
plt.show()

print(f'Width:   min={min(widths)}  max={max(widths)}  mean={np.mean(widths):.0f}')
print(f'Height:  min={min(heights)}  max={max(heights)}  mean={np.mean(heights):.0f}')

In [ ]:
# Preprocessing preview — show pipeline stages on one image
from utils.preprocess import upscale, to_gray, bilateral_denoise, adaptive_threshold, morphological_clean

img = cv2.imread(str(images[0]))
up    = upscale(img)
gray  = to_gray(up)
den   = bilateral_denoise(gray)
thr   = adaptive_threshold(den)
clean = morphological_clean(thr)

stages = [
    ('Original', cv2.cvtColor(img, cv2.COLOR_BGR2RGB)),
    ('Upscaled', cv2.cvtColor(up, cv2.COLOR_BGR2RGB)),
    ('Grayscale', gray),
    ('Denoised', den),
    ('Threshold', thr),
    ('Clean', clean),
]

fig, axes = plt.subplots(1, len(stages), figsize=(18, 3))
for ax, (name, im) in zip(axes, stages):
    cmap = 'gray' if len(im.shape) == 2 else None
    ax.imshow(im, cmap=cmap)
    ax.set_title(name, fontsize=10)
    ax.axis('off')
plt.suptitle('Preprocessing pipeline', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Quick OCR sanity check on a few images
# Requires EasyOCR installed
from utils.preprocess import preprocess_plate
from utils.ocr import PlateReader

reader = PlateReader(gpu=False)  # Set gpu=True if available
sample_ocr = random.sample(images, min(5, len(images)))

for p in sample_ocr:
    img = cv2.imread(str(p))
    processed = preprocess_plate(img)
    result = reader.read(processed)
    print(f'{p.name:40s} → {result or "[unreadable]"}')